# Vehicle registrations cleanup

In [21]:
from pathlib import Path
import re

import pandas as pd


ANALYSIS_DIR = Path.cwd()
RAW_DIR = ANALYSIS_DIR / "data" / "raw" / "vehicle-registrations"
OUTPUT_PATH = ANALYSIS_DIR / "data" / "vehicle-registrations.csv"

SOURCE_TABLE = "BIL53"
HTML_FILES = {
    "diesel": RAW_DIR / "diesel.html",
    "electric": RAW_DIR / "electric.html",
    "petrol": RAW_DIR / "petrol.html",
    "phev": RAW_DIR / "phev.html",
}

TIME_PATTERN = re.compile(r"^\d{4}M\d{2}$")

In [22]:
def _to_text(value: object) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def _row_has_data(row: pd.Series, time_columns: list[str]) -> bool:
    numeric = pd.to_numeric(row[time_columns], errors="coerce")
    return numeric.notna().any()


def _classify_header_level(has_data_flags: list[bool], index: int) -> str | None:
    next1 = has_data_flags[index + 1] if index + 1 < len(has_data_flags) else None
    next2 = has_data_flags[index + 2] if index + 2 < len(has_data_flags) else None
    next3 = has_data_flags[index + 3] if index + 3 < len(has_data_flags) else None

    if next1 is True or next1 == True:
        return "terms_of_use"
    if (next1 is False or next1 == False) and (next2 is True or next2 == True):
        return "vehicle_type"
    if (next1 is False or next1 == False) and (next2 is False or next2 == False) and (next3 is True or next3 == True):
        return "region"
    return None


def parse_vehicle_html(path: Path, fuel_key: str) -> pd.DataFrame:
    table = pd.read_html(path, header=1, flavor="lxml")[0]
    first_column = table.columns[0]
    table = table.rename(columns={first_column: "label"})

    time_columns = [col for col in table.columns if TIME_PATTERN.match(str(col))]
    has_data_flags = [_row_has_data(table.iloc[i], time_columns) for i in range(len(table))]

    current = {
        "region": None,
        "vehicle_type": None,
        "terms_of_use": None,
    }

    records = []
    for i, row in table.iterrows():
        label = _to_text(row["label"])
        if not label:
            continue

        if has_data_flags[i]:
            values = pd.to_numeric(row[time_columns], errors="coerce")
            for time_col, registrations in values.items():
                if pd.isna(registrations):
                    continue
                records.append(
                    {
                        "time": str(time_col),
                        "region": current["region"],
                        "vehicle_type": current["vehicle_type"],
                        "terms_of_use": current["terms_of_use"],
                        "propellant": fuel_key,
                        "registrations": int(registrations),
                        "source_file": path.name,
                        "source_table": SOURCE_TABLE,
                    }
                )
            continue

        level = _classify_header_level(has_data_flags, i)
        if level is None:
            continue

        if level == "region":
            current["region"] = label
            current["vehicle_type"] = None
            current["terms_of_use"] = None
        elif level == "vehicle_type":
            current["vehicle_type"] = label
            current["terms_of_use"] = None
        elif level == "terms_of_use":
            current["terms_of_use"] = label

    return pd.DataFrame.from_records(records)


def build_vehicle_registrations_dataset(files: dict[str, Path]) -> pd.DataFrame:
    parts = [parse_vehicle_html(path, fuel_key) for fuel_key, path in files.items()]
    combined = pd.concat(parts, ignore_index=True)

    text_columns = ["region", "vehicle_type", "terms_of_use", "propellant"]
    for col in text_columns:
        combined[col] = combined[col].astype(str).str.strip().str.lower()

    # Harmonize labels for downstream analysis.
    combined["vehicle_type"] = combined["vehicle_type"].str.replace(", total", "", regex=False)
    combined["terms_of_use"] = combined["terms_of_use"].replace(
        {
            "in households": "private",
            "in industries": "company",
        }
    )

    combined["year"] = combined["time"].str.slice(0, 4).astype(int)
    combined["month"] = combined["time"].str.slice(5, 7).astype(int)

    combined = combined.rename(
        columns={
            "region": "municipality",
            "terms_of_use": "use_case",
            "registrations": "number_of_registrations",
        }
    )

    combined = combined.drop(columns=["time", "source_table"])
    combined = combined.drop_duplicates()
    combined = combined.sort_values(
        by=["year", "month", "municipality", "vehicle_type", "use_case", "propellant"],
        kind="stable",
    ).reset_index(drop=True)

    ordered_columns = [
        "year",
        "month",
        "municipality",
        "vehicle_type",
        "use_case",
        "propellant",
        "number_of_registrations",
        "source_file",
    ]
    return combined[ordered_columns]

In [23]:
vehicle_registrations = build_vehicle_registrations_dataset(HTML_FILES)
vehicle_registrations.head()

,year,month,municipality,vehicle_type,use_case,propellant,number_of_registrations,source_file
0,2018,1,aabenraa,lorries,company,diesel,0,diesel.html
1,2018,1,aabenraa,lorries,company,electric,0,electric.html
2,2018,1,aabenraa,lorries,company,petrol,0,petrol.html
3,2018,1,aabenraa,lorries,company,phev,0,phev.html
4,2018,1,aabenraa,lorries,private,diesel,0,diesel.html


In [24]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
vehicle_registrations.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH

PosixPath('/Users/marcuslauridsen/Developer/Github/social-data-final-project/analysis/data/vehicle-registrations.csv')

In [25]:
quality_report = {
    "rows": len(vehicle_registrations),
    "municipalities": vehicle_registrations["municipality"].nunique(),
    "vehicle_types": vehicle_registrations["vehicle_type"].nunique(),
    "use_cases": vehicle_registrations["use_case"].nunique(),
    "propellants": vehicle_registrations["propellant"].nunique(),
    "year_min": vehicle_registrations["year"].min(),
    "year_max": vehicle_registrations["year"].max(),
    "month_min": vehicle_registrations["month"].min(),
    "month_max": vehicle_registrations["month"].max(),
}

null_report = vehicle_registrations.isna().sum().to_dict()
quality_report, null_report

({'rows': 316800,
  'municipalities': 100,
  'vehicle_types': 4,
  'use_cases': 2,
  'propellants': 4,
  'year_min': np.int64(2018),
  'year_max': np.int64(2026),
  'month_min': np.int64(1),
  'month_max': np.int64(12)},
 {'year': 0,
  'month': 0,
  'municipality': 0,
  'vehicle_type': 0,
  'use_case': 0,
  'propellant': 0,
  'number_of_registrations': 0,
  'source_file': 0})

In [26]:
unique_municipalities = sorted(vehicle_registrations["municipality"].dropna().unique().tolist())

start_row = vehicle_registrations[["year", "month"]].sort_values(["year", "month"]).iloc[0]
end_row = vehicle_registrations[["year", "month"]].sort_values(["year", "month"]).iloc[-1]

time_coverage = {
    "start_period": f"{int(start_row['year'])}-M{int(start_row['month']):02d}",
    "end_period": f"{int(end_row['year'])}-M{int(end_row['month']):02d}",
}

print(f"Unique municipalities ({len(unique_municipalities)}):")
print(unique_municipalities)

print("\nTime coverage:")
print(time_coverage)

Unique municipalities (100):
['aabenraa', 'aalborg', 'aarhus', 'albertslund', 'allerød', 'assens', 'ballerup', 'billund', 'bornholm', 'brøndby', 'brønderslev', 'christiansø', 'copenhagen', 'dragør', 'egedal', 'esbjerg', 'faaborg-midtfyn', 'fanø', 'favrskov', 'faxe', 'fredensborg', 'fredericia', 'frederiksberg', 'frederikshavn', 'frederikssund', 'furesø', 'gentofte', 'gladsaxe', 'glostrup', 'greve', 'gribskov', 'guldborgsund', 'haderslev', 'halsnæs', 'hedensted', 'helsingør', 'herlev', 'herning', 'hillerød', 'hjørring', 'holbæk', 'holstebro', 'horsens', 'hvidovre', 'høje-taastrup', 'hørsholm', 'ikast-brande', 'ishøj', 'jammerbugt', 'kalundborg', 'kerteminde', 'kolding', 'køge', 'langeland', 'lejre', 'lemvig', 'lolland', 'lyngby-taarbæk', 'læsø', 'mariagerfjord', 'middelfart', 'morsø', 'norddjurs', 'nordfyns', 'not stated', 'nyborg', 'næstved', 'odder', 'odense', 'odsherred', 'randers', 'rebild', 'ringkøbing-skjern', 'ringsted', 'roskilde', 'rudersdal', 'rødovre', 'samsø', 'silkeborg', '

In [27]:
excluded_municipalities = ["christiansø", "not stated"]

excluded_rows = vehicle_registrations[
    vehicle_registrations["municipality"].isin(excluded_municipalities)
].copy()

excluded_summary = (
    excluded_rows.groupby("municipality", as_index=False)["number_of_registrations"]
    .agg(["count", "sum"])
    .reset_index()
)

print("Rows and registrations for municipalities to exclude:")
print(excluded_summary if not excluded_summary.empty else "No matching municipalities found.")

vehicle_registrations = vehicle_registrations[
    ~vehicle_registrations["municipality"].isin(excluded_municipalities)
].copy()

print(f"\nRows after exclusion: {len(vehicle_registrations)}")

Rows and registrations for municipalities to exclude:
   index municipality  count   sum
0      0  christiansø   3168     5
1      1   not stated   3168  1080

Rows after exclusion: 310464


In [28]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
vehicle_registrations.to_csv(OUTPUT_PATH, index=False)

final_quality_report = {
    "rows": len(vehicle_registrations),
    "municipalities": vehicle_registrations["municipality"].nunique(),
    "year_min": vehicle_registrations["year"].min(),
    "year_max": vehicle_registrations["year"].max(),
    "month_min": vehicle_registrations["month"].min(),
    "month_max": vehicle_registrations["month"].max(),
}

final_quality_report, OUTPUT_PATH

({'rows': 310464,
  'municipalities': 98,
  'year_min': np.int64(2018),
  'year_max': np.int64(2026),
  'month_min': np.int64(1),
  'month_max': np.int64(12)},
 PosixPath('/Users/marcuslauridsen/Developer/Github/social-data-final-project/analysis/data/vehicle-registrations.csv'))